In [12]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from folium.plugins import MarkerCluster

In [13]:
ciudad_lineal = '2807915'

hortaleza = '2807916'

In [14]:
df_renta = (
    pd.read_csv('/Users/david/Desktop/renta/31097.csv', sep = ';', dtype = 'category', usecols = ['Secciones','Periodo','Total'])
    .pipe(lambda df_:df_[~df_['Secciones'].isna()])
    .pipe(lambda df_:df_[df_['Periodo'] == '2022'])
    .drop(columns = ['Periodo'])
    .assign(Secciones = lambda df_:df_['Secciones'].str.split(' ').str[0])
    .assign(Total = lambda df_:(df_['Total'].astype('str').str.replace('.','').fillna('0.0')))
    .assign(Total = lambda df_:np.select([df_['Total'] == ''],['0'],df_['Total']))
    .assign(Total = lambda df_:df_['Total'].fillna('0').replace({'nan':'0'}).astype('int'))
    .astype({'Total':'float'})
    .groupby(['Secciones'], as_index=False, observed=True)
    .agg({'Total':'mean'})
    .round(2)
)

In [15]:
gdf = (
    gpd.read_file('/Users/david/Desktop/renta/seccionado_2024/SECC_CE_20240101.shp')
    .filter(['CUSEC','geometry'])
    .pipe(lambda df_:df_[df_['CUSEC'].str.startswith((hortaleza, ciudad_lineal))])
    .merge(df_renta, left_on = ['CUSEC'], right_on = ['Secciones'], how = 'left')
    .filter(['CUSEC','Total','geometry'])
    .to_crs('EPSG:4326')
)

In [16]:
geojson_data = gdf.to_json()

# Create a folium map centered on the data
m = folium.Map(location=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom_start=10)

# Add Choropleth layer
folium.Choropleth(
    geo_data=geojson_data,
    name="Choropleth",
    data=gdf,
    columns=["CUSEC", "Total"],
    key_on="feature.properties.CUSEC",
    fill_color="YlGnBu",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Renta anual",
).add_to(m)

# Save map
m.save("../docs/map.html")

/var/folders/97/8rcpw5b562zgpndp9bqs30bc0000gn/T/ipykernel_39726/2065358734.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  m = folium.Map(location=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom_start=10)
